# SpecStory Prompt Miner + Representation Learning (Colab / GPU)

This notebook does two things end to end:

1. **Mine + extract** real developer prompts from public GitHub repos that contain SpecStory session logs (`.specstory/history/*.md`).
2. **Learn representations** of those prompts on GPU: sentence embeddings, clustering, and 2D visualization.

The mining/extraction steps are network-bound (GitHub API). The **GPU is used in the embedding and clustering steps**, which is the machine-learning core of the project.

**Before running:** set `Runtime -> Change runtime type -> GPU`.

**You can either:**
- run discovery + extraction here to build a fresh dataset, **or**
- skip to *Step 8b* and upload an `extracted_prompts_*.csv` you already produced locally.

## Step 1 — Install dependencies

In [ ]:
!pip -q install sentence-transformers umap-learn scikit-learn pandas requests matplotlib

## Step 2 — Confirm the GPU is available

In [ ]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("CUDA available. GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("No GPU detected. Set Runtime -> Change runtime type -> GPU, then re-run.")

## Step 3 — GitHub tokens + config (supports up to 5 tokens)

More tokens = more requests before you hit a rate limit. The pool **rotates to the next token automatically** when one is throttled, and only sleeps when *all* tokens are exhausted. This is what lets us mine far more repos/prompts.

Provide tokens any of these ways (all are merged + de-duplicated):
- Colab secrets named `GITHUB_TOKEN`, `GITHUB_TOKEN_1` … `GITHUB_TOKEN_5` (key icon in the left sidebar) — **recommended**
- environment variables of the same names
- paste them comma-separated at the prompt when this cell runs

In [ ]:
import os, re, time, base64
from datetime import datetime
from functools import lru_cache
from pathlib import Path
from dataclasses import dataclass
from getpass import getpass

import requests
import pandas as pd

# Names checked in both environment variables and Colab secrets.
TOKEN_ENV_VARS = [
    "GITHUB_TOKEN", "GH_TOKEN",
    "GITHUB_TOKEN_1", "GITHUB_TOKEN_2", "GITHUB_TOKEN_3",
    "GITHUB_TOKEN_4", "GITHUB_TOKEN_5",
]


def _collect_tokens():
    found = []
    for var in TOKEN_ENV_VARS:
        val = os.getenv(var)
        if val:
            found.append(val.strip())
    try:
        from google.colab import userdata
        for var in TOKEN_ENV_VARS:
            try:
                val = userdata.get(var)
                if val:
                    found.append(val.strip())
            except Exception:
                pass
    except Exception:
        pass
    seen, unique = set(), []
    for tok in found:
        if tok and tok not in seen:
            seen.add(tok)
            unique.append(tok)
    return unique


_tokens = _collect_tokens()
if not _tokens:
    raw = getpass("Paste up to 5 GitHub tokens, comma-separated (blank = anon): ").strip()
    _tokens = [t.strip() for t in raw.split(",") if t.strip()]


class TokenPool:
    """Round-robin pool of GitHub tokens with rate-limit aware rotation."""

    def __init__(self, tokens):
        self.tokens = list(tokens) if tokens else [None]
        self.idx = 0

    @property
    def has_token(self):
        return any(t for t in self.tokens)

    def __len__(self):
        return len(self.tokens)

    def headers(self):
        token = self.tokens[self.idx]
        h = {"Accept": "application/vnd.github+json"}
        if token:
            h["Authorization"] = f"Bearer {token}"
        return h

    def rotate(self):
        self.idx = (self.idx + 1) % len(self.tokens)
        return self.tokens[self.idx]


TOKEN_POOL = TokenPool(_tokens)
BASE_URL = "https://api.github.com"

if TOKEN_POOL.has_token:
    n_real = len([t for t in TOKEN_POOL.tokens if t])
    print(f"Loaded {n_real} GitHub token(s). Code search enabled; "
          f"effective rate limit scales with token count.")
else:
    print("No tokens: repo search only, low rate limits, code search disabled.")

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

## Step 4 — Discovery queries (SpecStory + other prompt sources)

SpecStory is the highest-signal source of **real session prompts**, but it's not the only place people leave the prompts they actually typed. This notebook also mines:

| Family | GitHub filter(s) | What you get | Kind |
|--------|------------------|--------------|------|
| **SpecStory** | `path:.specstory/history`, `specstory in:readme`, `topic:specstory` | Full Cursor/agent chat logs — real user turns | `session_prompt` |
| **Aider** | `path:.aider.chat.history.md`, `filename:.aider.input.history` | Exactly what the dev typed to Aider (input history is verbatim prompts) | `session_prompt` |
| **Instruction files** | `filename:CLAUDE.md`, `filename:AGENTS.md`, `filename:.cursorrules`, `filename:copilot-instructions.md`, `path:.cursor/rules` | Reusable, hand-written prompts people use to steer agents | `instruction_file` / `rules_config` |

Other sources you *could* add later (not on by default, lower signal or noisier): `path:.continue/sessions`, Cline/Roo task histories, ChatGPT `conversations.json` exports, and inline `# Prompt:` / `// Prompt:` code comments.

Toggle families with `INCLUDE_AIDER` / `INCLUDE_INSTRUCTIONS` below.

In [ ]:
# --- Session-prompt artifacts: real human prompts inside chat logs ---
SPECSTORY_REPO_QUERIES = [
    ("specstory_readme", "specstory in:readme stars:>0"),
    ("topic_specstory", "topic:specstory"),
]
SPECSTORY_CODE_QUERIES = [
    ("specstory_history", "path:.specstory/history"),
]

# Aider records the exact inputs the developer typed.
AIDER_CODE_QUERIES = [
    ("aider_chat_history", "path:.aider.chat.history.md"),
    ("aider_input_history", "filename:.aider.input.history"),
]

# Reusable instruction prompts people write to steer agents.
INSTRUCTION_CODE_QUERIES = [
    ("claude_md", "filename:CLAUDE.md"),
    ("agents_md", "filename:AGENTS.md"),
    ("cursorrules", "filename:.cursorrules"),
    ("copilot_instructions", "filename:copilot-instructions.md"),
    ("cursor_rules", "path:.cursor/rules"),
]

# --- Toggle which artifact families to mine (SpecStory always on) ---
INCLUDE_AIDER = True
INCLUDE_INSTRUCTIONS = True

REPO_DETECTION_QUERIES = list(SPECSTORY_REPO_QUERIES)
CODE_SEARCH_QUERIES = list(SPECSTORY_CODE_QUERIES)
if INCLUDE_AIDER:
    CODE_SEARCH_QUERIES += AIDER_CODE_QUERIES
if INCLUDE_INSTRUCTIONS:
    CODE_SEARCH_QUERIES += INSTRUCTION_CODE_QUERIES

print(f"Repo-search queries: {len(REPO_DETECTION_QUERIES)} | "
      f"Code-search queries: {len(CODE_SEARCH_QUERIES)}")

## Step 5 — GitHub API helpers (search + pagination)

In [ ]:
GITHUB_SEARCH_RESULT_CAP = 1000


def _gh_get(url, params=None, timeout=30):
    """GET with multi-token rotation.

    Rotates to the next token on a rate-limit response and only sleeps once
    every token in the pool has been tried. Returns the final Response object
    (callers check status_code) or None on repeated network errors.
    """
    n = len(TOKEN_POOL)
    rate_hits = 0
    net_errors = 0
    while True:
        try:
            response = requests.get(url, headers=TOKEN_POOL.headers(), params=params, timeout=timeout)
        except requests.RequestException:
            net_errors += 1
            if net_errors >= 3:
                return None
            time.sleep(3)
            continue

        if response.status_code in (403, 429):
            rate_hits += 1
            if rate_hits < n:
                TOKEN_POOL.rotate()
                continue
            reset_ts = int(response.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = min(max(reset_ts - time.time() + 2, 15), 120)
            print(f"   All {n} token(s) limited. Waiting {int(wait)}s...", flush=True)
            time.sleep(wait)
            rate_hits = 0
            TOKEN_POOL.rotate()
            continue

        return response


def _search_page(endpoint, query, *, page, per_page):
    params = {"q": query, "sort": "updated", "order": "desc", "per_page": per_page, "page": page}
    response = _gh_get(f"{BASE_URL}/{endpoint}", params=params)
    if response is None:
        return None
    if response.status_code == 200:
        return response.json()
    print(f"   Error {response.status_code}: {response.text[:120]}", flush=True)
    return None


def _search_paginated(endpoint, query, max_results=1000, *, page_delay=2.0):
    per_page = 100
    max_results = min(max_results, GITHUB_SEARCH_RESULT_CAP)
    all_items, total_count, page = [], 0, 1
    while len(all_items) < max_results:
        payload = _search_page(endpoint, query, page=page, per_page=per_page)
        if not payload:
            break
        if page == 1:
            total_count = int(payload.get("total_count", 0))
        items = payload.get("items", [])
        if not items:
            break
        all_items.extend(items[: max_results - len(all_items)])
        if len(items) < per_page or len(all_items) >= max_results:
            break
        page += 1
        if page * per_page > GITHUB_SEARCH_RESULT_CAP:
            break
        time.sleep(page_delay)
    return all_items, total_count


def _repo_record(repo, query, discovery_source, match_url=None, matched_path="", matched_file=""):
    created_at = repo.get("created_at") or repo.get("pushed_at") or repo.get("updated_at")
    age_months = 0.0
    if created_at:
        try:
            created = datetime.strptime(created_at, "%Y-%m-%dT%H:%M:%SZ")
            age_months = (datetime.utcnow() - created).days / 30
        except ValueError:
            age_months = 0.0
    repo_url = repo.get("html_url") or f"https://github.com/{repo.get('full_name', '')}"
    return {
        "full_name": repo.get("full_name"),
        "url": repo_url,
        "match_url": match_url or repo_url,
        "description": (repo.get("description") or "")[:120],
        "language": repo.get("language"),
        "stars": repo.get("stargazers_count", 0),
        "forks": repo.get("forks_count", 0),
        "open_issues": repo.get("open_issues_count", 0),
        "size_kb": repo.get("size", 0),
        "created_at": repo.get("created_at"),
        "updated_at": repo.get("updated_at"),
        "age_months": round(age_months, 1),
        "topics": ", ".join(repo.get("topics", []) or []),
        "detection_query": query,
        "discovery_source": discovery_source,
        "matched_path": matched_path,
        "matched_file": matched_file,
    }


@lru_cache(maxsize=8192)
def _fetch_repo_details(full_name):
    response = _gh_get(f"{BASE_URL}/repos/{full_name}")
    if response is not None and response.status_code == 200:
        return response.json()
    return {"full_name": full_name, "html_url": f"https://github.com/{full_name}"}


def search_repos(query, max_results=1000):
    items, total = _search_paginated("search/repositories", query, max_results=max_results)
    return [_repo_record(r, query, "repo_search") for r in items], total


def search_code(query, max_results=1000):
    if not TOKEN_POOL.has_token:
        print(f"   skipping code search for '{query}' (needs a token)", flush=True)
        return [], 0
    items, total = _search_paginated("search/code", query, max_results=max_results)
    repos = []
    for item in items:
        repository = item.get("repository")
        if not repository:
            continue
        details = _fetch_repo_details(repository["full_name"])
        repos.append(
            _repo_record(
                details, query, "code_search",
                match_url=item.get("html_url"),
                matched_path=item.get("path", ""),
                matched_file=item.get("name", ""),
            )
        )
    return repos, total

## Step 6 — SpecStory parser + prompt extraction

This is the improved parser that recognizes modern SpecStory speaker markers (`_**User (timestamp)**_`, `_**Agent (...)**_`), stops user turns cleanly at the next non-user speaker, strips `<think>`/`<tool-use>`/HTML-comment noise, and is CJK-aware. It mirrors `src/github_repo_miner/prompt_extraction.py`.

In [ ]:
TARGET_ROOTS = [
    ".specstory/history",
    ".aider.chat.history.md",
    ".aider.input.history",
    ".cursor/rules",
    ".github/copilot-instructions.md",
    "copilot-instructions.md",
    "CLAUDE.md",
    "AGENTS.md",
    ".cursorrules",
]

INSTRUCTION_FILE_NAMES = {"copilot-instructions.md", "claude.md", "agents.md"}
RULES_PATH_FRAGMENTS = (".cursor/rules", ".cursorrules")
SPECSTORY_PATH_FRAGMENT = ".specstory/history"
AIDER_CHAT_FILE = ".aider.chat.history.md"
AIDER_INPUT_FILE = ".aider.input.history"

ARTIFACT_PRIORITY = {
    "session_prompt": 1,
    "inline_prompt_comment": 2,
    "instruction_file": 3,
    "rules_config": 4,
}

_ROLE_WORDS = r"(User|Human|Developer|Assistant|Agent|AI|System|Tool)"
SPEAKER_PATTERNS = (
    re.compile(r"^\s*#{1,6}\s*\*{0,3}\s*" + _ROLE_WORDS + r"\b.*$", re.IGNORECASE),
    re.compile(r"^\s*_{0,3}\*{1,3}\s*" + _ROLE_WORDS + r"\b.*$", re.IGNORECASE),
    re.compile(r"^\s*" + _ROLE_WORDS + r"\s*:\s*.*$", re.IGNORECASE),
)
ROLE_COLON_RE = re.compile(r"^\s*" + _ROLE_WORDS + r"\s*:\s*(.*)$", re.IGNORECASE)
USER_ROLES = {"user", "human", "developer"}
PROMPT_FIELD_RE = re.compile(r"^\s*(prompt|query|request|instruction)\s*:\s*(.+?)\s*$", re.IGNORECASE)
CJK_RE = re.compile(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]")

_HTML_COMMENT_RE = re.compile(r"<!--.*?-->", re.DOTALL)
_THINK_BLOCK_RE = re.compile(r"<think\b.*?</think>", re.DOTALL | re.IGNORECASE)
_TOOL_BLOCK_RE = re.compile(r"<tool-use\b.*?</tool-use>", re.DOTALL | re.IGNORECASE)
_SEPARATOR_LINES = {"---", "***", "___"}

INLINE_PROMPT_PATTERNS = (
    re.compile(r"^\s*#\s*Prompt:\s*(.+?)\s*$", re.IGNORECASE),
    re.compile(r"^\s*//\s*Prompt:\s*(.+?)\s*$", re.IGNORECASE),
    re.compile(r"^\s*/\*\s*Prompt:\s*(.+?)\s*\*/\s*$", re.IGNORECASE),
)


def _normalize_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n").strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def _speaker_role(line):
    for pattern in SPEAKER_PATTERNS:
        match = pattern.match(line)
        if match:
            return match.group(1).lower()
    return None


def _inline_after_role(line):
    match = ROLE_COLON_RE.match(line)
    return match.group(2).strip() if match else ""


def _has_cjk(text):
    return bool(CJK_RE.search(text))


def _clean_turn(lines):
    text = "\n".join(lines)
    text = _HTML_COMMENT_RE.sub("", text)
    text = _THINK_BLOCK_RE.sub("", text)
    text = _TOOL_BLOCK_RE.sub("", text)
    kept = [ln for ln in text.splitlines() if ln.strip() not in _SEPARATOR_LINES]
    return _normalize_text("\n".join(kept))


def _is_meaningful_prompt(text):
    stripped = text.strip()
    if len(stripped) < 5:
        return False
    if len(stripped.split()) >= 3:
        return True
    return _has_cjk(stripped) and len(stripped) >= 6


def _dedupe_meaningful(prompts):
    """Drop fragments and de-duplicate while preserving order."""
    seen, cleaned = set(), []
    for prompt in prompts:
        if not _is_meaningful_prompt(prompt) or prompt in seen:
            continue
        seen.add(prompt)
        cleaned.append(prompt)
    return cleaned


@dataclass
class PromptRecord:
    full_name: str
    repo_url: str
    source_file: str
    artifact_type: str
    research_priority: int
    prompt_index: int
    prompt_text: str
    extraction_method: str
    source_line: "int | None" = None


def _request_json(url, params=None):
    response = _gh_get(url, params=params)
    if response is not None and response.status_code == 200:
        return response.json()
    return None


@lru_cache(maxsize=1024)
def _fetch_repo(full_name):
    data = _request_json(f"{BASE_URL}/repos/{full_name}")
    return data if isinstance(data, dict) else None


def classify_artifact(path):
    lower = path.lower().replace("\\", "/")
    name = Path(lower).name
    if SPECSTORY_PATH_FRAGMENT in lower:
        return "session_prompt", "specstory_turn_parser"
    if name == AIDER_CHAT_FILE:
        return "session_prompt", "aider_chat_parser"
    if name == AIDER_INPUT_FILE:
        return "session_prompt", "aider_input_parser"
    if lower == ".cursorrules" or any(f in lower for f in RULES_PATH_FRAGMENTS):
        return "rules_config", "rules_file"
    if name in INSTRUCTION_FILE_NAMES or ".github/copilot-instructions.md" in lower:
        return "instruction_file", "instruction_file"
    if lower.endswith(".md") and "prompt" in lower:
        return "rules_config", "rules_file"
    return "instruction_file", "instruction_file"


def _is_prompt_file(path):
    lower = path.lower().replace("\\", "/")
    artifact_type, _ = classify_artifact(path)
    return artifact_type in {"session_prompt", "instruction_file", "rules_config"} or (
        lower.endswith(".md") and "prompt" in lower
    )


def _walk_contents(full_name, path, depth=0, max_depth=2):
    url = f"{BASE_URL}/repos/{full_name}/contents/{path}".rstrip("/")
    payload = _request_json(url)
    if payload is None:
        return []
    if isinstance(payload, dict):
        return [payload] if payload.get("type") == "file" else []
    items = []
    for entry in payload:
        if entry.get("type") == "file":
            items.append(entry)
        elif entry.get("type") == "dir" and depth < max_depth:
            items.extend(_walk_contents(full_name, entry["path"], depth + 1, max_depth))
    return items


def discover_prompt_files(full_name):
    files = {}
    for root in TARGET_ROOTS:
        for entry in _walk_contents(full_name, root):
            path = entry.get("path") or ""
            if path and _is_prompt_file(path):
                files[path] = entry
    return list(files.values())


def _decode_file_content(payload):
    content = payload.get("content")
    if not content:
        return ""
    try:
        return base64.b64decode(content.replace("\n", "")).decode("utf-8", errors="replace")
    except Exception:
        return ""


def fetch_file_text(full_name, path):
    payload = _request_json(f"{BASE_URL}/repos/{full_name}/contents/{path}")
    if not isinstance(payload, dict):
        return ""
    return _normalize_text(_decode_file_content(payload))


def parse_specstory_prompts(text):
    prompts, current, capturing, saw_marker = [], [], False, False

    def flush():
        nonlocal current
        candidate = _clean_turn(current)
        current = []
        if candidate:
            prompts.append(candidate)

    for raw_line in text.splitlines():
        role = _speaker_role(raw_line)
        if role is not None:
            saw_marker = True
            if capturing and current:
                flush()
            capturing = role in USER_ROLES
            current = []
            if capturing:
                inline = _inline_after_role(raw_line)
                if inline:
                    current.append(inline)
            continue
        if capturing:
            current.append(raw_line)
    if capturing and current:
        flush()

    if not saw_marker:
        for raw_line in text.splitlines():
            match = PROMPT_FIELD_RE.match(raw_line)
            if match:
                prompts.append(_normalize_text(match.group(2)))

    return _dedupe_meaningful(prompts)


def parse_aider_chat_history(text):
    """Aider .aider.chat.history.md: user turns are lines prefixed with '#### '."""
    prompts, current = [], []

    def flush():
        nonlocal current
        joined = _normalize_text("\n".join(current))
        current = []
        if joined:
            prompts.append(joined)

    for raw_line in text.splitlines():
        if raw_line.startswith("####"):
            current.append(raw_line[4:].lstrip())
        elif current:
            flush()
    if current:
        flush()
    return _dedupe_meaningful(prompts)


def parse_aider_input_history(text):
    """Aider .aider.input.history: user inputs are lines prefixed with '+'."""
    prompts, current = [], []

    def flush():
        nonlocal current
        joined = _normalize_text("\n".join(current))
        current = []
        if joined:
            prompts.append(joined)

    for raw_line in text.splitlines():
        if raw_line.startswith("+"):
            current.append(raw_line[1:])
        elif current:
            flush()
    if current:
        flush()
    return _dedupe_meaningful(prompts)


def _record_from_parts(*, full_name, repo_url, source_file, artifact_type,
                       extraction_method, prompt_index, prompt_text, source_line=None):
    return PromptRecord(
        full_name=full_name, repo_url=repo_url, source_file=source_file,
        artifact_type=artifact_type, research_priority=ARTIFACT_PRIORITY[artifact_type],
        prompt_index=prompt_index, prompt_text=prompt_text,
        extraction_method=extraction_method, source_line=source_line,
    )


def extract_prompts_from_repo(full_name, repo_url=None):
    repo = _fetch_repo(full_name)
    if not repo:
        return []
    repo_url = repo_url or repo.get("html_url") or f"https://github.com/{full_name}"
    records = []
    for entry in discover_prompt_files(full_name):
        path = entry.get("path") or ""
        if not path:
            continue
        text = fetch_file_text(full_name, path)
        if not text:
            continue
        artifact_type, extraction_method = classify_artifact(path)
        if extraction_method == "specstory_turn_parser":
            prompts = parse_specstory_prompts(text)
        elif extraction_method == "aider_chat_parser":
            prompts = parse_aider_chat_history(text)
        elif extraction_method == "aider_input_parser":
            prompts = parse_aider_input_history(text)
        else:
            prompts = [_normalize_text(text)]
        for index, prompt in enumerate(prompts, start=1):
            records.append(_record_from_parts(
                full_name=full_name, repo_url=repo_url, source_file=path,
                artifact_type=artifact_type, extraction_method=extraction_method,
                prompt_index=index, prompt_text=prompt,
            ))
    return records


def _apply_artifact_filters(df, *, max_priority=None, artifact_types=None):
    filtered = df.copy()
    if max_priority is not None:
        filtered = filtered[filtered["research_priority"] <= max_priority]
    if artifact_types:
        allowed = {v.strip() for v in artifact_types if v.strip()}
        filtered = filtered[filtered["artifact_type"].isin(allowed)]
    return filtered.reset_index(drop=True)


def _print_extraction_summary(df):
    print("\nArtifact breakdown:", flush=True)
    if df.empty:
        print("  (no rows)", flush=True)
        return
    for artifact_type, count in df["artifact_type"].value_counts().sort_index().items():
        print(f"  priority {ARTIFACT_PRIORITY[artifact_type]} | {artifact_type}: {count}", flush=True)


def extract_prompts_from_candidates(candidate_csv, output_dir="outputs", max_repos=None,
                                    max_priority=None, artifact_types=None):
    candidate_path = Path(candidate_csv)
    df = pd.read_csv(candidate_path)
    if df.empty:
        raise ValueError(f"No candidate rows found in {candidate_path}")
    if max_repos is not None:
        df = df.head(max_repos).copy()

    seen, rows = set(), []
    for index, row in enumerate(df.itertuples(index=False), start=1):
        full_name = getattr(row, "full_name")
        if full_name in seen:
            continue
        seen.add(full_name)
        repo_url = getattr(row, "url", None)
        print(f"[{index:03d}/{len(df)}] extracting from {full_name}", flush=True)
        for record in extract_prompts_from_repo(full_name, repo_url=repo_url):
            rows.append({
                "full_name": record.full_name, "repo_url": record.repo_url,
                "source_file": record.source_file, "source_line": record.source_line,
                "artifact_type": record.artifact_type, "research_priority": record.research_priority,
                "prompt_index": record.prompt_index, "prompt_text": record.prompt_text,
                "prompt_length": len(record.prompt_text),
                "prompt_word_count": len(record.prompt_text.split()),
                "extraction_method": record.extraction_method,
            })

    extracted_df = pd.DataFrame(rows)
    _print_extraction_summary(extracted_df)
    filtered_df = _apply_artifact_filters(extracted_df, max_priority=max_priority, artifact_types=artifact_types)
    if len(filtered_df) != len(extracted_df):
        print(f"\nApplied artifact filter: {len(extracted_df)} -> {len(filtered_df)} rows", flush=True)
        _print_extraction_summary(filtered_df)

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    ts = pd.Timestamp.utcnow().strftime("%Y%m%d_%H%M")
    output_file = output_path / f"extracted_prompts_{candidate_path.stem}_{ts}.csv"
    filtered_df.to_csv(output_file, index=False)
    print(f"\nExtracted prompts: {len(filtered_df)}", flush=True)
    print(f"Saved prompt dataset: {output_file.name}", flush=True)
    return output_file

## Step 7 — Run discovery (build the candidate repo list)

`MAX_RESULTS_PER_QUERY` is set to GitHub's hard cap of **1000 results per query**. With more queries (SpecStory + Aider + instruction files) and the 5-token pool, this finds far more repos. This is network-bound, not GPU-bound.

In [ ]:
MAX_RESULTS_PER_QUERY = 1000  # GitHub's hard cap per search query


def run_discovery(max_results_per_query=MAX_RESULTS_PER_QUERY):
    rows = []
    for name, q in REPO_DETECTION_QUERIES:
        print(f"[repo] {name}: {q}", flush=True)
        repos, total = search_repos(q, max_results=max_results_per_query)
        print(f"   got {len(repos)} (total on github: {total})", flush=True)
        rows.extend(repos)
    for name, q in CODE_SEARCH_QUERIES:
        print(f"[code] {name}: {q}", flush=True)
        repos, total = search_code(q, max_results=max_results_per_query)
        print(f"   got {len(repos)} (total on github: {total})", flush=True)
        rows.extend(repos)
    return pd.DataFrame(rows).drop_duplicates(subset="full_name").reset_index(drop=True)


candidates_df = run_discovery()
candidate_csv = OUTPUT_DIR / f"specstory_candidates_{pd.Timestamp.utcnow().strftime('%Y%m%d_%H%M')}.csv"
candidates_df.to_csv(candidate_csv, index=False)
print(f"\nCandidate repos: {len(candidates_df)} -> {candidate_csv.name}")
candidates_df.head()

## Step 8 — Extract prompts from candidates (ALL repos)

`MAX_REPOS = None` runs over **every** discovered repo, and `ARTIFACT_TYPES = None` keeps **all** prompt kinds (SpecStory + Aider session prompts, plus instruction/rules files). This is the slow, network-bound part — with many repos it can run for a long time, but the 5-token pool keeps it from stalling on rate limits.

The saved CSV contains every artifact type. Step 9 narrows to `session_prompt` for the ML path, so your raw gathered data stays complete here.

In [ ]:
MAX_REPOS = None        # None = ALL discovered repos
ARTIFACT_TYPES = None   # None = keep every artifact type (session + instruction + rules)

prompts_path = extract_prompts_from_candidates(
    candidate_csv,
    output_dir=OUTPUT_DIR,
    max_repos=MAX_REPOS,
    artifact_types=ARTIFACT_TYPES,
)
print("Saved:", prompts_path)

## Step 8b (alternative) — Upload an existing extracted CSV

Skip Steps 7–8 and run this instead if you already have an `extracted_prompts_*.csv` from a local run.

In [ ]:
# Run ONLY if you want to upload your own CSV instead of mining above.
from google.colab import files

uploaded = files.upload()
prompts_path = list(uploaded.keys())[0]
print("Using uploaded file:", prompts_path)

## Step 9 — Load & clean the prompt dataset

In [ ]:
prompts_df = pd.read_csv(prompts_path)
print("Rows loaded:", len(prompts_df))

if "artifact_type" in prompts_df.columns:
    prompts_df = prompts_df[prompts_df["artifact_type"] == "session_prompt"].copy()

prompts_df = prompts_df.dropna(subset=["prompt_text"])
prompts_df["prompt_text"] = prompts_df["prompt_text"].astype(str).str.strip()
prompts_df = prompts_df[prompts_df["prompt_text"].str.len() >= 10]
prompts_df = prompts_df.drop_duplicates(subset="prompt_text").reset_index(drop=True)

texts = prompts_df["prompt_text"].tolist()
print("Unique prompts to embed:", len(texts))
prompts_df.head()

## Step 10 — Embed prompts on GPU

Uses `all-MiniLM-L6-v2` sentence embeddings. This is the GPU-accelerated step.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Embeddings shape:", embeddings.shape)

## Step 11 — Cluster prompts (pick k by silhouette)

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

candidate_ks = [k for k in range(4, 21) if k < len(texts)]
best_k, best_score, best_labels = None, -1.0, None

for k in candidate_ks:
    km = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)
    print(f"k={k:2d}  silhouette={score:.4f}")
    if score > best_score:
        best_k, best_score, best_labels = k, score, labels

if best_labels is None:
    best_k, best_labels = 1, np.zeros(len(texts), dtype=int)

print(f"\nBest k = {best_k} (silhouette = {best_score:.4f})")
prompts_df["cluster"] = best_labels

## Step 12 — Visualize clusters (UMAP 2D)

In [ ]:
import umap
import matplotlib.pyplot as plt

n_neighbors = min(15, max(2, len(texts) - 1))
reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=0.1, metric="cosine", random_state=42)
coords = reducer.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
sc = plt.scatter(coords[:, 0], coords[:, 1], c=best_labels, cmap="tab20", s=10, alpha=0.75)
plt.title(f"SpecStory prompts — {best_k} clusters (UMAP)")
plt.colorbar(sc, label="cluster")
plt.tight_layout()
plt.savefig("outputs/prompt_clusters_umap.png", dpi=150)
plt.show()

## Step 13 — Inspect clusters (sample prompts per cluster)

In [ ]:
for c in sorted(prompts_df["cluster"].unique()):
    sub = prompts_df[prompts_df["cluster"] == c]
    print(f"\n=== Cluster {c}  ({len(sub)} prompts) ===")
    for t in sub["prompt_text"].head(5):
        snippet = t.replace("\n", " ")
        print("  -", (snippet[:160] + "...") if len(snippet) > 160 else snippet)

## Step 14 — Save & download results

In [ ]:
clustered_csv = "outputs/specstory_prompts_clustered.csv"
prompts_df.to_csv(clustered_csv, index=False)
np.save("outputs/prompt_embeddings.npy", embeddings)
print("Saved:", clustered_csv, "and outputs/prompt_embeddings.npy")

from google.colab import files
files.download(clustered_csv)
files.download("outputs/prompt_clusters_umap.png")